In [20]:
# Standard Libraries
import math
import os
import pickle
import random
import time
from datetime import datetime

# Data Manipulation and Visualization
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
matplotlib.use('Agg')  # For saving figures
from IPython.display import clear_output
from tqdm import tqdm

# PyTorch Libraries and Tools
import torch
import torch.nn as nn
from torch.autograd import Variable
from torch.utils.tensorboard import SummaryWriter

from modules import Autoencoder, GAN, Discriminator, MINE
from modules.utils import convert_ipynb_to_html  # For saving HTML files
import importlib  # For reloading modules
importlib.reload(Autoencoder)
importlib.reload(GAN)
importlib.reload(Discriminator)
importlib.reload(MINE)

<module 'modules.MINE' from 'd:\\mingyu\\coding\\github\\infoQGAN\\modules\\MINE.py'>

In [21]:
# Function to calculate total trainable parameters
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# Print total number of trainable parameters
total_params = count_parameters(generator)
print(f"Total trainable parameters in LinearGenerator: {total_params}")

Total trainable parameters in LinearGenerator: 116


In [24]:
# 전역 변수 선언
train_type = "InfoGAN"
use_mine = True if train_type == "InfoGAN" else False
DIGITS_STR = "0123456789"
DIGITS = [0,1,2,3,4,5,6,7,8,9]
TARGETS_STR = "35"
TARGETS = [3,5]
G_lr = 0.005
M_lr = 0.00005
D_lr = 0.002
smooth = 0.15
SEED_R = 1.7
SEED_DIM = 10
code_dim = 3
epoch_num = 500
gamma = 0.5
latent_dim = 16
num_images_per_class = 2000
COEFF = 2
ARGS = None

In [25]:
# 1. autoencoder 모델 준비
autoencoder = Autoencoder.Autoencoder(latent_dim=latent_dim)
autoencoder_epochs = 100
autoencoder_lr = 0.0001
autoencoder_coeff = 0.0005
autoencoder.load_state_dict(torch.load(f'savepoints/InfomaxEncoder_{DIGITS_STR}_{latent_dim}_ep{autoencoder_epochs}_lr{autoencoder_lr}_{autoencoder_coeff}.pth', weights_only=True))
autoencoder.eval()  # 평가 모드로 전환

# 2. 데이터 로드
data = np.load(f'./data/MNIST/{DIGITS_STR}_{latent_dim}_{autoencoder_epochs}_{autoencoder_lr}_{autoencoder_coeff}/mnist_{DIGITS_STR}_{latent_dim}_{num_images_per_class}.npz')

In [ ]:
# 4. 학습 준비:
    # 학습 데이터, 테스트 데이터, 검증 데이터를 2:1:1로 나눈다.
    # cpu/gpu 설정 및 device설정
    # n_qubits, code_qubits, noise_qubits, output_qubits 설정
print("이번 학습으로 생성할 숫자는", TARGETS, "입니다.")
# 원래는 DIGIT 하나만이었는데, 이제는 TARGETS 내부 숫자들을 모두 학습해야 한다.
train_dataset = np.concatenate([data[f'{target}_latent'][:num_images_per_class//4] for target in TARGETS], axis=0)
test_dataset = np.concatenate([data[f'{target}_latent'][num_images_per_class//2:num_images_per_class*3//4] for target in TARGETS], axis=0)
val_dataset = np.concatenate([data[f'{target}_latent'][num_images_per_class*3//4:] for target in TARGETS], axis=0)
train_size, test_size, val_size = len(train_dataset), len(test_dataset), len(val_dataset)
print("train_size =", train_size, "test_size =", test_size, "val_size =", val_size)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("학습에 사용할 device =",device)

# 5. 생성자, 판별자, MINE, optimizer 초기화
generator_initial_params = Variable(torch.tensor(np.random.normal(-np.pi/3, np.pi/3, (n_layers, n_qubits, 1))), requires_grad=True)
generator = GAN.LinearGenerator(input_dim=8, output_dim=16, hidden_size=4)
discriminator = Discriminator.LinearDiscriminator(input_dim = latent_dim, hidden_size=100) # 50 --> 25 변경
mine = MINE.LinearMine(code_dim=code_dim, output_dim=latent_dim, size=100) # 50 --> 100 변경
print("n_qubits = {} n_layers = {} 총 파라미터 수 = {}".format(n_qubits, n_layers, generator_initial_params.numel()))

G_opt = torch.optim.Adam([generator.params], lr=G_lr)
D_opt = torch.optim.Adam(discriminator.parameters(), lr=D_lr)
M_opt = torch.optim.Adam(mine.parameters(), lr=M_lr)

G_scheduler = torch.optim.lr_scheduler.StepLR(G_opt, step_size=30, gamma=gamma)
D_scheduler = torch.optim.lr_scheduler.StepLR(D_opt, step_size=30, gamma=gamma)
M_scheduler = torch.optim.lr_scheduler.StepLR(M_opt, step_size=30, gamma=gamma)
generator = 